# Detección de piezas faltantes y fuera de posición con OpenCV

## Objetivo del ejercicio

En este ejercicio simularemos una inspección automática de charolas en un proceso de manufactura. Cada charola debe contener 12 piezas organizadas en una matriz de 3 filas por 4 columnas. El sistema de visión deberá revisar cada posición y decidir si el ensamble está:

- **Correcto:** existe una pieza en la posición esperada.
- **Faltante:** no se encontró la pieza que debía ocupar esa posición.
- **Fuera de posición:** existe una pieza cercana, pero está desplazada respecto a su ubicación esperada.

El propósito es aprender un caso muy frecuente en manufactura: no basta con contar objetos; también hay que verificar que cada componente esté presente y colocado en el sitio correcto.

## Contexto de manufactura

Imaginemos una estación donde una cámara observa una charola antes de que el producto continúe a la siguiente operación. Las posiciones están previamente definidas. Una pieza puede estar perfectamente fabricada y aun así causar un problema si falta o si quedó desplazada.

La solución seguirá este flujo:

1. Generar o capturar una imagen de la charola.
2. Separar las piezas del fondo mediante color.
3. Encontrar cada pieza con contornos.
4. Comparar el centro detectado contra las posiciones esperadas.
5. Asignar un estado a cada posición.
6. Emitir una decisión de calidad para la charola completa.

### Supuestos del ejemplo

El fondo y la iluminación son relativamente estables, las piezas no se ocultan unas a otras y la separación entre posiciones es suficiente para distinguir un desplazamiento. El dataset es sintético, se genera con una semilla fija y queda guardado dentro del entorno temporal de Colab.

## 1. Preparar las bibliotecas y parámetros

`opencv-python-headless` contiene las funciones de visión artificial. `numpy` permite crear las imágenes como matrices de píxeles. `pandas` ayuda a organizar las etiquetas y las predicciones, mientras que `matplotlib` se utiliza para inspeccionar visualmente los resultados.

La semilla aleatoria es importante: si se vuelve a ejecutar la celda, se reconstruye el mismo dataset. Los parámetros de la charola y de las piezas están visibles para que el estudiante pueda modificarlos y observar cómo cambian los resultados.

In [ ]:
!pip -q install opencv-python-headless pandas matplotlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
IMAGE_WIDTH, IMAGE_HEIGHT = 640, 420
ROWS, COLUMNS = 3, 4
DATASET_SIZE = 60
PIECE_RADIUS = 22
POSITION_TOLERANCE = 24
MAX_DISPLACEMENT_DISTANCE = 85
dataset_dir = Path('/content/dataset_charolas_manufactura')
dataset_dir.mkdir(parents=True, exist_ok=True)
print('OpenCV:', cv2.__version__)
print(f'Cada imagen tendrá {ROWS * COLUMNS} posiciones esperadas.')

## 2. Definir la geometría esperada de la charola

Antes de detectar, necesitamos conocer dónde debería estar cada componente. La función siguiente crea 12 coordenadas de referencia en forma de cuadrícula. En una aplicación real, estas coordenadas podrían obtenerse durante la calibración de la cámara o a partir de un patrón fijo de la estación.

La separación horizontal y vertical es mayor que el diámetro de una pieza. Esto facilita que un desplazamiento sea distinguible de una pieza colocada correctamente. El resultado es una tabla de posiciones que se usará tanto para crear el dataset como para evaluar el algoritmo.

In [ ]:
def create_expected_positions():
    expected = []
    x_positions = np.linspace(170, 470, COLUMNS).astype(int)
    y_positions = np.linspace(100, 320, ROWS).astype(int)
    position_id = 0
    for row, center_y in enumerate(y_positions):
        for column, center_x in enumerate(x_positions):
            expected.append({'position_id': position_id, 'row': row, 'column': column, 'expected_x': center_x, 'expected_y': center_y})
            position_id += 1
    return pd.DataFrame(expected)

expected_positions = create_expected_positions()
display(expected_positions)

plt.figure(figsize=(9, 5))
plt.scatter(expected_positions.expected_x, expected_positions.expected_y, s=140, facecolors='none', edgecolors='blue')
for _, position in expected_positions.iterrows():
    plt.text(position.expected_x, position.expected_y, str(position.position_id), ha='center', va='center')
plt.gca().invert_yaxis(); plt.xlim(80, 560); plt.ylim(390, 30)
plt.title('Posiciones esperadas de los 12 componentes'); plt.xlabel('Coordenada X'); plt.ylabel('Coordenada Y'); plt.grid(alpha=.3); plt.show()

## 3. Generar el dataset incluido

Esta celda crea 60 imágenes sintéticas. Cada imagen comienza con una charola gris y marcas oscuras que representan los espacios de colocación. Para cada una de las 12 posiciones se sortea un estado: aproximadamente 65% serán correctas, 20% tendrán la pieza faltante y 15% tendrán la pieza desplazada.

Las etiquetas se guardan en `ground_truth.csv`. Para cada posición se registra el estado real y, cuando hay pieza visible, sus coordenadas. El dataset está incluido conceptualmente en el notebook porque la receta exacta de generación y las etiquetas se encuentran aquí; al ejecutar la celda se materializan los archivos en `/content/dataset_charolas_manufactura`.

In [ ]:
ground_truth_rows = []

for image_id in range(DATASET_SIZE):
    image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH, 3), (135, 135, 135), dtype=np.uint8)
    image = np.clip(image.astype(np.int16) + rng.normal(0, 3, image.shape), 0, 255).astype(np.uint8)
    # Dibujar visualmente los 12 espacios de la charola.
    for _, position in expected_positions.iterrows():
        expected_center = (int(position.expected_x), int(position.expected_y))
        cv2.circle(image, expected_center, PIECE_RADIUS + 7, (75, 75, 75), 2)

    for _, position in expected_positions.iterrows():
        random_value = rng.random()
        if random_value < 0.20:
            state = 'faltante'
            actual_x, actual_y = np.nan, np.nan
        elif random_value < 0.35:
            state = 'fuera_de_posicion'
            shift_x = int(rng.choice([-1, 1]) * rng.integers(38, 58))
            shift_y = int(rng.choice([-1, 1]) * rng.integers(20, 42))
            actual_x = int(position.expected_x + shift_x)
            actual_y = int(position.expected_y + shift_y)
            cv2.circle(image, (actual_x, actual_y), PIECE_RADIUS, (55, 190, 70), -1)
        else:
            state = 'correcta'
            actual_x, actual_y = int(position.expected_x), int(position.expected_y)
            cv2.circle(image, (actual_x, actual_y), PIECE_RADIUS, (55, 190, 70), -1)
        if state != 'faltante':
            cv2.circle(image, (actual_x - 7, actual_y - 7), 4, (220, 220, 220), -1)
        ground_truth_rows.append({'image_id': image_id, 'position_id': int(position.position_id), 'expected_x': int(position.expected_x), 'expected_y': int(position.expected_y), 'state': state, 'actual_x': actual_x, 'actual_y': actual_y})
    cv2.imwrite(str(dataset_dir / f'charola_{image_id:03d}.png'), image)

ground_truth = pd.DataFrame(ground_truth_rows)
ground_truth.to_csv(dataset_dir / 'ground_truth.csv', index=False)
print(f'Dataset creado: {DATASET_SIZE} imágenes, {len(ground_truth)} posiciones etiquetadas.')
display(ground_truth.head(12))
display(ground_truth.state.value_counts().rename_axis('estado').to_frame('cantidad'))

## 4. Inspeccionar ejemplos del dataset

Mirar las imágenes antes de ejecutar el detector es una práctica importante. Permite verificar si el problema está bien definido y si las etiquetas tienen sentido visualmente. Las piezas verdes son componentes presentes; los círculos grises sin pieza verde representan posiciones faltantes; una pieza verde alejada del centro de su círculo representa un componente fuera de posición.

La conversión BGR a RGB solo se usa para mostrar correctamente los colores con matplotlib. OpenCV lee las imágenes en BGR.

In [ ]:
sample_ids = [0, 1, 2, 3, 4, 5]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, image_id in zip(axes.ravel(), sample_ids):
    image_bgr = cv2.imread(str(dataset_dir / f'charola_{image_id:03d}.png'))
    axis.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axis.set_title(f'Charola {image_id}')
    axis.axis('off')
plt.suptitle('Muestras del dataset incluido', fontsize=16); plt.tight_layout(); plt.show()

## 5. Segmentar las piezas por color

El detector no necesita analizar todos los píxeles de la misma manera. Primero creamos una máscara binaria donde el blanco significa 'posible pieza verde' y el negro significa 'fondo o marca de la charola'.

HSV resulta útil porque el canal H representa el tono. El rango `[35, 90]` captura los verdes de las piezas. Los canales S y V evitan aceptar grises poco saturados o zonas demasiado oscuras. Después aplicamos una operación morfológica de cierre para unir pequeñas interrupciones provocadas por el brillo dibujado sobre cada pieza.

In [ ]:
def build_piece_mask(image_bgr):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 60, 40])
    upper_green = np.array([90, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    kernel = np.ones((5, 5), np.uint8)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

example_image = cv2.imread(str(dataset_dir / 'charola_000.png'))
example_mask = build_piece_mask(example_image)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(cv2.cvtColor(example_image, cv2.COLOR_BGR2RGB)); axes[0].set_title('Imagen original'); axes[0].axis('off')
axes[1].imshow(example_mask, cmap='gray'); axes[1].set_title('Máscara de piezas verdes'); axes[1].axis('off')
plt.show()

## 6. Detectar centros y revisar cada posición

`findContours` encuentra cada región blanca de la máscara. El centro de la caja delimitadora es una aproximación al centro de la pieza.

Para cada posición esperada se busca la detección más cercana. Si no hay ninguna dentro de `MAX_DISPLACEMENT_DISTANCE`, se considera que falta la pieza. Si la detección está dentro de `POSITION_TOLERANCE`, la posición es correcta. Si está más lejos, pero todavía puede asociarse razonablemente a esa posición, se marca como fuera de posición.

La función también evita reutilizar la misma detección para dos posiciones. Esto es importante: una pieza no puede justificar simultáneamente dos espacios de la charola.

In [ ]:
def detect_centers(image_bgr):
    mask = build_piece_mask(image_bgr)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for contour in contours:
        if cv2.contourArea(contour) < 200:
            continue
        x, y, width, height = cv2.boundingRect(contour)
        detections.append({'detected_x': x + width / 2, 'detected_y': y + height / 2, 'x': x, 'y': y, 'width': width, 'height': height})
    return pd.DataFrame(detections)

def inspect_tray(image_bgr, image_id):
    detections = detect_centers(image_bgr)
    used_indices = set()
    results = []
    annotated = image_bgr.copy()
    for _, position in expected_positions.iterrows():
        best_index, best_distance = None, np.inf
        for detection_index, detection in detections.iterrows():
            if detection_index in used_indices:
                continue
            distance = np.hypot(detection.detected_x - position.expected_x, detection.detected_y - position.expected_y)
            if distance < best_distance:
                best_index, best_distance = detection_index, distance
        if best_index is None or best_distance > MAX_DISPLACEMENT_DISTANCE:
            state = 'faltante'
            actual_x, actual_y = np.nan, np.nan
            box_color = (0, 0, 255)
        else:
            used_indices.add(best_index)
            detection = detections.loc[best_index]
            actual_x, actual_y = detection.detected_x, detection.detected_y
            state = 'correcta' if best_distance <= POSITION_TOLERANCE else 'fuera_de_posicion'
            box_color = (0, 180, 0) if state == 'correcta' else (0, 165, 255)
            cv2.rectangle(annotated, (int(detection.x), int(detection.y)), (int(detection.x + detection.width), int(detection.y + detection.height)), box_color, 2)
        center = (int(position.expected_x), int(position.expected_y))
        cv2.circle(annotated, center, 8, box_color, 2)
        cv2.putText(annotated, state, (center[0] - 38, center[1] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.45, box_color, 2)
        results.append({'image_id': image_id, 'position_id': int(position.position_id), 'predicted_state': state, 'detected_x': actual_x, 'detected_y': actual_y, 'distance_to_expected': best_distance if np.isfinite(best_distance) else np.nan})
    return annotated, pd.DataFrame(results)

annotated_example, example_result = inspect_tray(example_image, 0)
display(example_result)
plt.figure(figsize=(13, 7)); plt.imshow(cv2.cvtColor(annotated_example, cv2.COLOR_BGR2RGB)); plt.title('Inspección de una charola'); plt.axis('off'); plt.show()

## 7. Ejecutar la inspección sobre todas las imágenes

Esta celda representa el proceso repetitivo de una estación de visión. Se inspeccionan las 60 charolas, se guarda el estado predicho para cada una de sus 12 posiciones y se conserva una copia anotada de algunas imágenes para revisar visualmente el comportamiento.

El resultado esperado es una fila por posición y por imagen: 60 × 12 = 720 decisiones. Esto es más informativo que producir únicamente una decisión por imagen, porque permite saber exactamente qué posición falla.

In [ ]:
all_predictions = []
annotated_examples = []
for image_id in range(DATASET_SIZE):
    image_bgr = cv2.imread(str(dataset_dir / f'charola_{image_id:03d}.png'))
    annotated, tray_predictions = inspect_tray(image_bgr, image_id)
    all_predictions.append(tray_predictions)
    if image_id < 6:
        annotated_examples.append(annotated)
predictions = pd.concat(all_predictions, ignore_index=True)
print(f'Decisiones generadas: {len(predictions)}')
display(predictions.head(12))

## 8. Evaluar la decisión por posición

Unimos la predicción con la etiqueta real usando `image_id` y `position_id`. Después calculamos la matriz de confusión. Cada fila representa el estado real y cada columna el estado decidido por OpenCV.

La exactitud global indica qué porcentaje de las 720 posiciones fue clasificado correctamente. También calculamos métricas por clase: el recall de `faltante` nos dice qué proporción de las posiciones realmente faltantes fue detectada como faltante; la precisión de `faltante` indica cuántas de las posiciones rechazadas realmente estaban vacías.

In [ ]:
evaluation = ground_truth.merge(predictions, on=['image_id', 'position_id'], how='left')
evaluation['is_correct'] = evaluation.state == evaluation.predicted_state
accuracy = evaluation.is_correct.mean()
confusion_matrix = pd.crosstab(evaluation.state, evaluation.predicted_state, margins=True)
print(f'Exactitud global por posición: {accuracy:.1%}')
display(confusion_matrix)

class_metrics = []
for class_name in ['correcta', 'faltante', 'fuera_de_posicion']:
    true_positive = ((evaluation.state == class_name) & (evaluation.predicted_state == class_name)).sum()
    actual_count = (evaluation.state == class_name).sum()
    predicted_count = (evaluation.predicted_state == class_name).sum()
    class_metrics.append({'clase': class_name, 'casos_reales': actual_count, 'casos_predichos': predicted_count, 'recall': true_positive / actual_count if actual_count else 0, 'precision': true_positive / predicted_count if predicted_count else 0})
display(pd.DataFrame(class_metrics).round(3))

## 8.1. Resumen visible de métricas

Esta celda adicional reúne los indicadores principales en una tabla compacta. Se coloca inmediatamente después de la evaluación por posición para que las métricas aparezcan antes de la decisión global de la charola. Así, al ejecutar el notebook de arriba hacia abajo, el lector puede identificar claramente cuántos casos se analizaron y qué tan bien funcionó el detector.

También se muestra el porcentaje de cada estado real. Esto es importante porque un conjunto con muy pocas piezas faltantes puede producir una exactitud global alta aunque el sistema no sea bueno detectando faltantes.

In [ ]:
metric_summary = pd.DataFrame({
    'Indicador': [
        'Imágenes analizadas',
        'Posiciones esperadas analizadas',
        'Exactitud global por posición',
        'Recall de piezas faltantes',
        'Precisión de piezas faltantes',
        'Recall de piezas fuera de posición',
        'Precisión de piezas fuera de posición',
        'Exactitud de charola completa'
    ],
    'Resultado': [
        f'{evaluation.image_id.nunique():,}',
        f'{len(evaluation):,}',
        f'{accuracy:.1%}',
        f"{class_metrics[1]['recall']:.1%}",
        f"{class_metrics[1]['precision']:.1%}",
        f"{class_metrics[2]['recall']:.1%}",
        f"{class_metrics[2]['precision']:.1%}",
        'Se calculará en la siguiente sección'
    ]
})
display(metric_summary)

print('Distribución real de las posiciones:')
display((evaluation.state.value_counts(normalize=True).mul(100).round(1).rename('porcentaje').to_frame()))

## 9. Convertir las posiciones en una decisión de charola

En producción normalmente se necesita una decisión sencilla para el operador o para un PLC. Una regla conservadora será: la charola completa es `APROBADA` solamente si sus 12 posiciones fueron clasificadas como correctas. Si al menos una posición está faltante o fuera de posición, la charola será `RECHAZADA`.

Esta regla es deliberadamente estricta. En un proceso real podría existir una tercera salida, como `REVISIÓN MANUAL`, para casos con baja confianza o con una detección ambigua.

In [ ]:
tray_decisions = evaluation.groupby('image_id').apply(lambda group: 'APROBADA' if (group.predicted_state == 'correcta').all() else 'RECHAZADA').rename('predicted_tray_decision').reset_index()
true_tray_decisions = ground_truth.groupby('image_id').apply(lambda group: 'APROBADA' if (group.state == 'correcta').all() else 'RECHAZADA').rename('true_tray_decision').reset_index()
tray_evaluation = true_tray_decisions.merge(tray_decisions, on='image_id')
tray_evaluation['is_correct'] = tray_evaluation.true_tray_decision == tray_evaluation.predicted_tray_decision
print(f'Exactitud de la decisión completa de charola: {tray_evaluation.is_correct.mean():.1%}')
display(pd.crosstab(tray_evaluation.true_tray_decision, tray_evaluation.predicted_tray_decision, margins=True))
display(tray_evaluation.head(10))

## 9.1. Resumen final de métricas

La siguiente celda actualiza el resumen incorporando la métrica de charola completa. Esta separación es intencional: primero medimos cada posición y después medimos la decisión integral de liberar o rechazar la charola.

In [ ]:
final_metric_summary = metric_summary.copy()
final_metric_summary.loc[final_metric_summary['Indicador'] == 'Exactitud de charola completa', 'Resultado'] = f"{tray_evaluation.is_correct.mean():.1%}"
display(final_metric_summary)

print('Interpretación rápida:')
print(f"- Se analizaron {len(evaluation):,} posiciones en {evaluation.image_id.nunique()} charolas.")
print(f"- La exactitud por posición fue {accuracy:.1%}.")
print(f"- La exactitud de la decisión completa de charola fue {tray_evaluation.is_correct.mean():.1%}.")
print('- Para liberar producto, debe revisarse especialmente el recall de las clases faltante y fuera_de_posicion.')

## 10. Visualizar resultados y distribución de fallas

Los rectángulos verdes representan piezas detectadas en la posición correcta. Los rectángulos naranjas representan piezas encontradas, pero desplazadas. Las marcas rojas indican posiciones que el algoritmo consideró faltantes.

La gráfica de distribución ayuda a responder una pregunta operativa: ¿qué tipo de problema ocurre con mayor frecuencia? Si predominan las piezas faltantes, conviene investigar alimentación, surtido o ensamblaje. Si predominan las piezas fuera de posición, conviene investigar guías, vibración, herramientas de colocación o la geometría de la charola.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, annotated in zip(axes.ravel(), annotated_examples):
    axis.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); axis.axis('off')
plt.suptitle('Resultados anotados de la inspección', fontsize=16); plt.tight_layout(); plt.show()

evaluation.state.value_counts().reindex(['correcta', 'faltante', 'fuera_de_posicion']).plot(kind='bar', color=['#2ca02c', '#d62728', '#ff9900'], figsize=(8, 4), title='Distribución real de estados')
plt.ylabel('Número de posiciones'); plt.xlabel('Estado'); plt.xticks(rotation=0); plt.show()

## Interpretación amplia de los resultados

### Interpretar la exactitud por posición

La exactitud global por posición responde cuántas de las 720 posiciones fueron evaluadas correctamente. Si el valor es 100%, el sistema distinguió correctamente las tres situaciones en todas las posiciones. Si es 95%, significa que aproximadamente 5 de cada 100 posiciones recibieron una decisión incorrecta.

La exactitud debe leerse junto con el número total de casos. Un porcentaje puede parecer alto cuando hay pocos ejemplos. Por eso el notebook muestra también `casos_reales` y `casos_predichos` para cada clase. Estos conteos indican si existe suficiente representación de piezas correctas, faltantes y desplazadas.

### Interpretar la matriz de confusión

Las celdas de la diagonal (`correcta/correcta`, `faltante/faltante` y `fuera_de_posicion/fuera_de_posicion`) son aciertos. Las celdas fuera de la diagonal son errores.

- `faltante / correcta`: el sistema creyó que había una pieza, aunque realmente no la había. Esto puede suceder si una marca, sombra o ruido se interpreta como componente. Es un **falso pase** si la charola se aprueba a partir de esa decisión.
- `correcta / faltante`: el sistema rechazó una posición que sí tenía una pieza correcta. Es un **falso rechazo** y puede provocar desperdicio o revisión manual innecesaria.
- `fuera_de_posicion / correcta`: la pieza estaba desplazada, pero el sistema aceptó el ensamble. Es especialmente importante porque un componente mal colocado puede producir una falla posterior aunque la pieza esté físicamente presente.
- `correcta / fuera_de_posicion`: la pieza estaba bien colocada, pero el sistema fue demasiado estricto o la imagen tenía una variación geométrica. Puede indicar que `POSITION_TOLERANCE` es demasiado pequeño.

La matriz permite decidir qué parámetro revisar. Muchos errores entre `correcta` y `fuera_de_posicion` sugieren revisar la tolerancia. Muchos errores entre `faltante` y `correcta` sugieren revisar la distancia máxima, la segmentación o el fondo de la charola.

In [ ]:
print('Métrica: matriz de confusión por posición')
display(confusion_matrix)

confusion_percent = pd.crosstab(evaluation.state, evaluation.predicted_state, normalize='index').mul(100).round(1)
print('Matriz de confusión expresada como porcentaje dentro de cada estado real:')
display(confusion_percent)
print('La diagonal representa aciertos; los valores fuera de la diagonal representan errores.')

### Interpretar precision y recall por clase

El **recall de faltantes** responde: de todas las posiciones que realmente estaban vacías, ¿cuántas identificamos? Un recall bajo significa que hay piezas faltantes que están pasando inadvertidas. En calidad, este error suele ser prioritario porque el producto incompleto puede avanzar al cliente o a la siguiente estación.

La **precisión de faltantes** responde: de todas las posiciones que marcamos como faltantes, ¿cuántas estaban realmente vacías? Una precisión baja genera demasiados rechazos falsos. El equipo de producción tendría que abrir muchas charolas correctas para confirmar que no existe un problema.

El recall de `fuera_de_posicion` mide la capacidad de encontrar componentes desplazados. La precisión de esa clase indica si las alarmas de desplazamiento son confiables. En ambos casos, la tolerancia representa una decisión de ingeniería: una tolerancia pequeña detecta movimientos sutiles, pero puede aumentar falsas alarmas; una tolerancia grande reduce falsas alarmas, pero puede dejar pasar desplazamientos peligrosos.

In [ ]:
print('Métricas por clase: precisión y recall')
display(pd.DataFrame(class_metrics).assign(recall_pct=lambda table: (table.recall * 100).round(1), precision_pct=lambda table: (table.precision * 100).round(1))[['clase', 'casos_reales', 'casos_predichos', 'recall_pct', 'precision_pct']].rename(columns={'recall_pct': 'recall (%)', 'precision_pct': 'precision (%)'}))
print('Recall: de los casos reales de una clase, cuántos encontró el sistema.')
print('Precisión: de las alarmas emitidas para una clase, cuántas fueron correctas.')

### Interpretar la decisión de la charola completa

La decisión de charola es más exigente que la decisión por posición. Aunque el sistema tenga alta exactitud individual, basta con que una de las 12 posiciones sea marcada como incorrecta para rechazar la charola. Por eso la exactitud de charola completa puede ser menor que la exactitud por posición.

Esta diferencia es normal y tiene una explicación sencilla: hay 12 oportunidades de error en cada charola. En un proceso real, la regla de aprobación debe definirse con calidad e ingeniería. Puede ser correcto rechazar cualquier charola con una posición incorrecta, pero también puede existir un proceso de revisión manual para reducir paros o desperdicio.

Un **falso pase de charola** ocurre cuando el sistema aprueba una charola que realmente tiene una pieza faltante o fuera de posición. Un **falso rechazo de charola** ocurre cuando el sistema rechaza una charola que en realidad estaba completa. El primero afecta directamente la conformidad del producto; el segundo afecta eficiencia, costo y capacidad de producción.

In [ ]:
print('Métricas de decisión de charola completa')
tray_accuracy = tray_evaluation.is_correct.mean()
tray_confusion = pd.crosstab(tray_evaluation.true_tray_decision, tray_evaluation.predicted_tray_decision, margins=True)
tray_metric_summary = pd.DataFrame({'Indicador': ['Charolas analizadas', 'Charolas realmente aprobadas', 'Charolas realmente rechazadas', 'Exactitud de charola completa'], 'Resultado': [len(tray_evaluation), (tray_evaluation.true_tray_decision == 'APROBADA').sum(), (tray_evaluation.true_tray_decision == 'RECHAZADA').sum(), f'{tray_accuracy:.1%}']})
display(tray_metric_summary)
print('Matriz de confusión de la decisión final:')
display(tray_confusion)
print('Un falso pase es una charola realmente RECHAZADA que el sistema marcó como APROBADA.')
print('Un falso rechazo es una charola realmente APROBADA que el sistema marcó como RECHAZADA.')

### Ejemplo de lectura para un supervisor

Si los resultados muestran recall de faltantes de 100% y precisión de faltantes de 92%, la solución está encontrando todos los espacios vacíos, pero algunas alarmas son falsas. Es una solución segura respecto a piezas faltantes, aunque puede generar revisiones adicionales.

Si el recall de faltantes fuera 80%, el sistema estaría dejando pasar 2 de cada 10 faltantes aproximadamente. Aunque la exactitud global fuera alta, esta situación no sería aceptable sin una revisión adicional, porque el indicador crítico no está protegiendo suficientemente contra el riesgo de producto incompleto.

Si el recall de fuera de posición es bajo, no necesariamente significa que OpenCV no detecte las piezas. Puede significar que la tolerancia es demasiado amplia y el algoritmo acepta como correcta una pieza que está desplazada. En cambio, si la precisión es baja, puede que la cámara tenga vibración, que la charola no esté alineada o que la tolerancia sea demasiado pequeña.

## Conclusiones generales detalladas

1. **La presencia no es suficiente.** Contar 12 piezas no garantiza que la charola sea correcta. La comparación contra posiciones esperadas permite detectar piezas faltantes y piezas desplazadas.

2. **OpenCV puede resolver problemas controlados de inspección.** En este ejemplo, una máscara de color y el análisis de contornos son suficientes porque las piezas contrastan claramente con el fondo. La solución es explicable: cada decisión puede relacionarse con una distancia entre el centro observado y el centro esperado.

3. **Los parámetros representan tolerancias de proceso.** `POSITION_TOLERANCE` no es solamente un detalle de programación; expresa cuánto movimiento se acepta como normal. Su valor debe definirse con planos, especificaciones y datos reales.

4. **Los errores tienen costos distintos.** Un falso pase puede liberar un producto incompleto, mientras que un falso rechazo puede generar desperdicio o detener la operación. La configuración correcta depende del riesgo de calidad y del costo operativo.

5. **La exactitud por posición y la exactitud por charola no son iguales.** Una charola con muchas posiciones ofrece más oportunidades de error. Por ello conviene reportar ambos niveles: detalle por componente y decisión final del ensamble.

6. **El dataset sintético sirve para aprender, no para certificar.** Las imágenes fueron generadas bajo condiciones conocidas. Para una implementación real habría que capturar datos de diferentes lotes, turnos, operadores, cámaras, niveles de iluminación, suciedad, vibraciones y orientaciones.

7. **La solución necesita una estrategia para casos dudosos.** En producción no todo debe reducirse a aprobado o rechazado. Se puede agregar una tercera salida de revisión manual cuando la distancia esté cerca del límite o cuando la máscara tenga una forma anormal.

8. **El siguiente paso sería validar con imágenes reales.** Con datos reales podrían ajustarse los rangos HSV, la distancia mínima entre componentes, la tolerancia de posición y los criterios de aceptación. Si el defecto no puede distinguirse por color, se podrían incorporar forma, textura o modelos de aprendizaje profundo.

En conclusión, este ejercicio muestra cómo convertir una imagen de una charola en una decisión operativa trazable. El valor principal no está solamente en dibujar cajas, sino en relacionar cada detección con una posición esperada y traducir el resultado a una acción de calidad.

## Cómo ejecutar en Google Colab

1. Abre [Google Colab](https://colab.research.google.com/).
2. Selecciona **Archivo → Subir notebook**.
3. Sube este archivo `.ipynb`.
4. Ejecuta las celdas en orden con `Shift + Enter`.
5. Revisa las imágenes, la matriz de confusión, las métricas por clase y la decisión final de cada charola.

No se requiere GPU ni ningún archivo adicional. El dataset se genera automáticamente dentro de la sesión de Colab.